In [3]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from itertools import combinations
import os

# -------------------------------------------------
# 1. READ THE THREE HUMAN EVALUATION FILES
# -------------------------------------------------

# Use the corrected evaluator 1 file
df1 = pd.read_excel("HUMAN_EVALUATION1.xlsx")
df2 = pd.read_excel("HUMAN_EVALUATION2.xlsx")
df3 = pd.read_excel("HUMAN_EVALUATION3.xlsx")

print("Files loaded successfully.")

# -------------------------------------------------
# 2. COLUMNS TO ANALYZE
# -------------------------------------------------

columns = {
    "ChatGPT Zero-shot": "ChatGPT ZeroShot_Hallucination",
    "ChatGPT CoT": "CoT_Hallucination ChatGPT",
    "ChatGPT Structured": "Structured_Hallucination  ChatGPT",

    "Llama Zero-shot": "ZeroShot_Hallucination LLama",
    "Llama CoT": "CoT_Hallucination",
    "Llama Structured": "Structured_Hallucination",

    "Gemini Zero-shot": "Zero-Shot Gemini",
    "Gemini CoT": "Chain-of-Thought Gemini",
    "Gemini Structured": "Structured Gemini"
}

# -------------------------------------------------
# VALIDATION CHECK 1
# Ensure every annotation is 1, 2 or 3
# -------------------------------------------------

valid_values = {1, 2, 3}

for file_name, df in [
    ("HUMAN_EVALUATION1.xlsx", df1),
    ("HUMAN_EVALUATION2.xlsx", df2),
    ("HUMAN_EVALUATION3.xlsx", df3)
]:

    for col in columns.values():

        invalid = df.loc[
            ~df[col].isin(valid_values),
            col
        ]

        if not invalid.empty:

            raise ValueError(
                f"\nInvalid value(s) found!\n"
                f"File: {file_name}\n"
                f"Column: {col}\n"
                f"Invalid values: {invalid.tolist()}"
            )

print("✓ Validation Check 1 Passed: All annotation values are valid.")

# -------------------------------------------------
# 3. CREATE MAJORITY-VOTE CONSENSUS
# -------------------------------------------------

consensus = pd.DataFrame()

for name, col in columns.items():

    ratings = pd.concat(
        [
            df1[col],
            df2[col],
            df3[col]
        ],
        axis=1
    )

    # Majority vote
    consensus[name] = ratings.mode(axis=1)[0]

print("Majority-vote consensus created successfully.")

# -------------------------------------------------
# VALIDATION CHECK 2
# Ensure category counts sum to total number of items
# -------------------------------------------------

total_items = len(consensus)

print("\nChecking category counts...\n")

for col in consensus.columns:

    counts = consensus[col].value_counts()

    count1 = counts.get(1, 0)
    count2 = counts.get(2, 0)
    count3 = counts.get(3, 0)

    total = count1 + count2 + count3

    print(
        f"{col}: "
        f"1={count1}, "
        f"2={count2}, "
        f"3={count3}, "
        f"Total={total}"
    )

    if total != total_items:

        raise ValueError(
            f"\nValidation Check 2 Failed!\n"
            f"{col} has {total} items instead of {total_items}."
        )

print(f"\n✓ Validation Check 2 Passed: Every configuration totals {total_items} items.")

# -------------------------------------------------
# SAVE CONSENSUS FILE
# -------------------------------------------------

consensus.to_excel(
    "Human_Evaluation_Consensus(Updated).xlsx",
    index=False
)

print("Human_Evaluation_Consensus(Updated).xlsx saved.")

# -------------------------------------------------
# 4. CONVERT TO BINARY
# -------------------------------------------------

# 3 = Correct / No Hallucination
# 1 or 2 = Hallucination

binary = (consensus == 3).astype(int)

# -------------------------------------------------
# 5. McNEMAR FUNCTION
# -------------------------------------------------

def run_mcnemar(name1, name2):

    a = binary[name1]
    b = binary[name2]

    table = pd.crosstab(a, b)

    table = table.reindex(
        index=[0,1],
        columns=[0,1],
        fill_value=0
    )

    result = mcnemar(
        table.values,
        exact=True
    )

    return {

        "Comparison": f"{name1} vs {name2}",

        "Both Incorrect (0,0)": table.loc[0,0],

        f"{name1} Incorrect, {name2} Correct": table.loc[0,1],

        f"{name1} Correct, {name2} Incorrect": table.loc[1,0],

        "Both Correct (1,1)": table.loc[1,1],

        "McNemar Statistic": result.statistic,

        "p-value": result.pvalue
    }

# -------------------------------------------------
# 6. RUN ALL PAIRWISE COMPARISONS
# -------------------------------------------------

results = []

for model1, model2 in combinations(columns.keys(), 2):

    results.append(
        run_mcnemar(model1, model2)
    )

results_df = pd.DataFrame(results)

print("\nMcNemar Test Results:\n")
display(results_df)

# -------------------------------------------------
# 7. SAVE McNEMAR RESULTS
# -------------------------------------------------

results_df.to_excel(
    "McNemar_Test_Results(Updated).xlsx",
    index=False
)

print("McNemar_Test_Results(Updated).xlsx saved successfully.")
print("Human_Evaluation_Consensu(Updated))s.xlsx saved successfully.")

Files loaded successfully.
✓ Validation Check 1 Passed: All annotation values are valid.
Majority-vote consensus created successfully.

Checking category counts...

ChatGPT Zero-shot: 1=46, 2=43, 3=311, Total=400
ChatGPT CoT: 1=50, 2=44, 3=306, Total=400
ChatGPT Structured: 1=46, 2=45, 3=309, Total=400
Llama Zero-shot: 1=49, 2=106, 3=245, Total=400
Llama CoT: 1=52, 2=113, 3=235, Total=400
Llama Structured: 1=49, 2=88, 3=263, Total=400
Gemini Zero-shot: 1=14, 2=8, 3=378, Total=400
Gemini CoT: 1=14, 2=7, 3=379, Total=400
Gemini Structured: 1=14, 2=7, 3=379, Total=400

✓ Validation Check 2 Passed: Every configuration totals 400 items.
Human_Evaluation_Consensus(Updated).xlsx saved.

McNemar Test Results:



,Comparison,"Both Incorrect (0,0)","ChatGPT Zero-shot Incorrect, ChatGPT CoT Correct","ChatGPT Zero-shot Correct, ChatGPT CoT Incorrect","Both Correct (1,1)",McNemar Statistic,p-value,"ChatGPT Zero-shot Incorrect, ChatGPT Structured Correct","ChatGPT Zero-shot Correct, ChatGPT Structured Incorrect","ChatGPT Zero-shot Incorrect, Llama Zero-shot Correct",...,"Llama Structured Incorrect, Gemini CoT Correct","Llama Structured Correct, Gemini CoT Incorrect","Llama Structured Incorrect, Gemini Structured Correct","Llama Structured Correct, Gemini Structured Incorrect","Gemini Zero-shot Incorrect, Gemini CoT Correct","Gemini Zero-shot Correct, Gemini CoT Incorrect","Gemini Zero-shot Incorrect, Gemini Structured Correct","Gemini Zero-shot Correct, Gemini Structured Incorrect","Gemini CoT Incorrect, Gemini Structured Correct","Gemini CoT Correct, Gemini Structured Incorrect"
0,ChatGPT Zero-shot vs ChatGPT CoT,89,0.0,5.0,306,0.0,6.250000e-02,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ChatGPT Zero-shot vs ChatGPT Structured,89,NaN,NaN,309,0.0,5.000000e-01,0.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ChatGPT Zero-shot vs Llama Zero-shot,70,NaN,NaN,226,19.0,3.785517e-11,NaN,NaN,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ChatGPT Zero-shot vs Llama CoT,72,NaN,NaN,218,17.0,7.237227e-14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ChatGPT Zero-shot vs Llama Structured,70,NaN,NaN,244,19.0,1.940957e-07,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ChatGPT Zero-shot vs Gemini Zero-shot,13,NaN,NaN,302,9.0,2.405361e-14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ChatGPT Zero-shot vs Gemini CoT,13,NaN,NaN,303,8.0,5.021389e-15,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ChatGPT Zero-shot vs Gemini Structured,13,NaN,NaN,303,8.0,5.021389e-15,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ChatGPT CoT vs ChatGPT Structured,90,NaN,NaN,305,1.0,3.750000e-01,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ChatGPT CoT vs Llama Zero-shot,73,NaN,NaN,224,21.0,1.069675e-09,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


McNemar_Test_Results(Updated).xlsx saved successfully.
Human_Evaluation_Consensu(Updated))s.xlsx saved successfully.
